In [ ]:
import hashlib
import time
import pickle

#Block class represents a single block in the blockchain
class Block:
    # Initialization
    def __init__(self, index, previous_hash):
        self.index = index
        self.timestamp = int(time.time())
        self.previous_hash = previous_hash
        self.transactions = []
        self.merkle_root = ""
        self.hash = ""

    # Calculate the Merkle root of the transactions in the block
    def calculate_merkle_root(self):
        if len(self.transactions) == 0:
            self.merkle_root = ""

        # Calculate the Merkle root using a simple approach
        hashes = [
            hashlib.sha256(tx.encode()).hexdigest()
            for tx in self.transactions
        ]
        
        while len(hashes) > 1:
            next_level = []
            
            for i in range(0, len(hashes), 2):
                left = hashes[i]
                right = hashes[i + 1] if i + 1 < len(hashes) else left
                combined_hash = hashlib.sha256((left + right).encode()).hexdigest()
                next_level.append(combined_hash)
            hashes = next_level

        self.merkle_root = hashes[0]
        return self.merkle_root

    # Calculate the hash
    def calculate_hash(self):
        self.calculate_merkle_root()
        block_string = pickle.dumps((
            self.index, self.timestamp, 
            self.previous_hash, self.merkle_root
        ))
        self.hash = hashlib.sha256(block_string).hexdigest()
        return self.hash


In [61]:
# Blockchain class represents the entire blockchain
class Blockchain:
    def __init__(self):
        self.chain = []
        self.create_block(transactions=["Genesis transaction"])

    # Create a new block and add it to the blockchain
    def create_block(self, transactions = None):
        # Init index and get previous hash
        index = len(self.chain) + 1
        if index == 1:
            previous_hash = "Genesis block"
        else:
            previous_hash = self.chain[-1].hash

        block = Block(index, previous_hash)

        # Set transactions if provided
        if transactions:
            block.transactions = transactions
        
        block.calculate_hash()

        if not self.validate_block(block):
            print("Invalid block. Cannot add to the chain.")
            return None
        else:
            self.chain.append(block)
            return block

    # Validate a block's integrity and its connection to the previous block
    def validate_block(self, block):
        # Genesis block validation
        if block.index == 1:
            return block.previous_hash == "Genesis block"

        # Check if the previous hash matches the last block's hash
        previous_block = self.chain[block.index - 2]
        previous_block.calculate_hash() 
        if block.previous_hash != previous_block.hash:
            print(f"Block {block.index} is invalid: previous hash not match.")
            return False

        return True

    # Validate the entire blockchain
    def validate_chain(self):
        for i in range(1, len(self.chain)):
            current_block = self.chain[i]
            valid = self.validate_block(current_block)
            if not valid:
                return False

        return True

In [81]:
import random

# Initialize a blockchain instance
blockchain = Blockchain()

# Add more 9 block
for i in range(9):
    # Initialize a list of transactions for the new block
    transactions = []
    for j in range(random.randint(1, 5)):
        transactions.append(
            f"Transfer {random.randint(1, 10)} coins " +
            f"from user{random.randint(1, 10)} to user{random.randint(1, 10)}"
        )

    # Create a new block with the generated transactions
    new_block = blockchain.create_block(transactions=transactions)

is_valid = blockchain.validate_chain()
print(f"Is chain valid: {is_valid}")

Is chain valid: True


In [82]:
import json

# Test: modify the transactions of the 5th block
block_to_modify = blockchain.chain[4]  # 5th block (index 4)
print("Block 5 before modification:\n" +
      f"{json.dumps(block_to_modify.__dict__, indent=2)}")

# Modify the transactions of the 5th block
block_to_modify.transactions[0] = "Transfer 100 coins from user1 to user2"
print("Block 5 after modification:\n"+
      f"{json.dumps(block_to_modify.__dict__, indent=2)}")

chain_valid_after_modification = blockchain.validate_chain()
print(f"Is chain valid: {chain_valid_after_modification}")

Block 5 before modification:
{
  "index": 5,
  "timestamp": 1787472618,
  "previous_hash": "1a908156af6faac807ffb91eda455e5adac1ec9bb371ee382b19837f7400d23c",
  "transactions": [
    "Transfer 5 coins from user6 to user9",
    "Transfer 4 coins from user3 to user5"
  ],
  "merkle_root": "dd8e0934f1e9c9e1c40718b8918d209464078e843c198b7b7be8e6f4d42a27a4",
  "hash": "662b260a8c283d15db448d3848d34e020620c1f1e191f523197de2d57a7aea72"
}
Block 5 after modification:
{
  "index": 5,
  "timestamp": 1787472618,
  "previous_hash": "1a908156af6faac807ffb91eda455e5adac1ec9bb371ee382b19837f7400d23c",
  "transactions": [
    "Transfer 100 coins from user1 to user2",
    "Transfer 4 coins from user3 to user5"
  ],
  "merkle_root": "dd8e0934f1e9c9e1c40718b8918d209464078e843c198b7b7be8e6f4d42a27a4",
  "hash": "662b260a8c283d15db448d3848d34e020620c1f1e191f523197de2d57a7aea72"
}
Block 6 is invalid: previous hash not match.
Is chain valid: False
